# [E1] Trusted Data Lab: Capa Plata, Limpieza y Puerta de Calidad
## Observatorio de Precios y Disponibilidad para Pescadería Valencia

| Metadato | Detalle |
| :--- | :--- |
| **Proyecto** | Observatorio de Precios y Disponibilidad de Productos del Mar |
| **Entrega** | [E1] Trusted Data Lab: Capa Plata, Limpieza y Puerta de Calidad |
| **Autor / PSP** | Felipe Sandoval (Data Engineering Lead) |
| **Cliente Operativo** | Pescadería Valencia (Colima / Manzanillo) |
| **Especies Foco** | Mojarra Fresca Entera, Huachinango del Pacífico, Filete de Pez Vela |
| **Arquitectura** | Medallion: Bronce -> Cuarentena / Plata -> Oro |


In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Suprimir advertencias de migración de Pandera
os.environ["DISABLE_PANDERA_IMPORT_WARNING"] = "True"

try:
    import pandera.pandas as pa
    from pandera.pandas import Column, Check, DataFrameSchema
except (ImportError, AttributeError):
    import pandera as pa
    from pandera import Column, Check, DataFrameSchema

from sklearn.preprocessing import MinMaxScaler, StandardScaler

print("Dependencias analíticas importadas correctamente.")


## 1. Ingesta de Capa Bronce y Resolución de Heterogeneidad

En esta etapa se cargan los datos crudos consolidados provenientes del SNIIM (precios mayoristas) y de las estaciones meteorológicas del litoral de Colima. Se resuelven heterogeneidades comunes:
- **Fechas:** Estandarización estricta al estándar internacional ISO 8601 (`YYYY-MM-DD`).
- **Unidades:** Precios expresados en Pesos Mexicanos por kilogramo (`$ MXN/kg`), precipitación en milímetros (`mm`), velocidad del viento en kilómetros por hora (`km/h`) y altura de oleaje en metros (`m`).
- **Tipado:** Coerción de tipos de datos flotantes y enteros, garantizando la eliminación de caracteres extraños.


In [ ]:
path_bronce = Path("../../data/bronce/precios_clima_historico_36p.csv")
if not path_bronce.exists():
    path_bronce = Path("data/bronce/precios_clima_historico_36p.csv")

df_bronce = pd.read_csv(path_bronce)
print(f"Total registros crudos leídos: {len(df_bronce)}")
display(df_bronce.head(3))


## 2. Definición del Contrato de Datos y Puerta de Calidad con Pandera

Para asegurar la confiabilidad del Trusted Data Lab, se implementa una **Puerta de Calidad (Quality Gate)** estricta mediante `Pandera`.
Los registros corruptos o anómalos **NO se descartan en silencio**:
1. Se evalúan contra reglas de dominio pesquero y rangos físicos.
2. Los registros infractores son aislados en `data/cuarentena/registros_invalidos.csv` con una columna explicativa `motivo_rechazo`.
3. Los registros aprobados continúan su flujo hacia la **Capa Plata**.


In [ ]:
PRODUCTOS_AUTORIZADOS = [
    "Mojarra Fresca Entera",
    "Huachinango del Pacífico",
    "Filete de Pez Vela"
]
ALERTAS_VALIDAS = ["NINGUNA", "AMARILLA", "NARANJA", "ROJA"]

schema_calidad = DataFrameSchema(
    columns={
        "periodo_id": Column(int, Check.greater_than_or_equal_to(1), nullable=False),
        "fecha_inicio_semana": Column(str, nullable=False),
        "mes": Column(int, Check.in_range(1, 12), nullable=False),
        "producto_foco": Column(str, Check.isin(PRODUCTOS_AUTORIZADOS), nullable=False),
        "precio_minimo_kg": Column(float, Check.greater_than(0), nullable=False),
        "precio_maximo_kg": Column(float, Check.greater_than(0), nullable=False),
        "precio_frecuente_kg": Column(float, Check.greater_than(0), nullable=False),
        "precipitacion_acumulada_mm": Column(float, Check.greater_than_or_equal_to(0), nullable=False),
        "velocidad_viento_kmh": Column(float, Check.in_range(0.0, 250.0), nullable=False),
        "altura_oleaje_m": Column(float, Check.in_range(0.0, 15.0), nullable=False),
        "dias_cierre_puerto": Column(int, Check.in_range(0, 7), nullable=False),
        "alerta_meteorologica": Column(str, Check.isin(ALERTAS_VALIDAS), nullable=False),
    },
    checks=[
        Check(
            lambda df: (df["precio_minimo_kg"] <= df["precio_frecuente_kg"] + 0.05) &
                       (df["precio_frecuente_kg"] <= df["precio_maximo_kg"] + 0.05),
            error="Inconsistencia de precios: min <= frecuente <= max"
        )
    ],
    coerce=True
)

print("Esquema Pandera compilado correctamente.")


## 3. Preprocesamiento Estadístico: Diagnóstico de Valores Faltantes e Imputación

En ingeniería de datos, el tratamiento de valores nulos exige diagnosticar el mecanismo estadístico subyacente:
- **MCAR (Missing Completely at Random):** Fallas aleatorias de comunicación con la estación meteorológica de Manzanillo. Tratamiento: **Interpolación temporal lineal** local a lo largo de la serie.
- **MAR (Missing at Random):** Datos faltantes de arribo o precios condicionados por factores observables (ej. cuando `dias_cierre_puerto > 0`, las embarcaciones ribereñas no zarpan). Tratamiento: **Imputación por mediana condicional** estratificada por `(producto_foco, temporada)`.
- **MNAR (Missing Not at Random):** Omisiones deliberadas en el reporte de precios durante colapsos de demanda. Tratamiento: Identificación mediante variables booleanas de control para no sesgar la varianza natural.


In [ ]:
df_plata = df_bronce.copy()

# Diagnóstico de nulos iniciales
nulos_iniciales = df_plata.isna().sum().sum()
print(f"Valores nulos iniciales en el lote: {nulos_iniciales}")

# Aplicación de Imputación Justificada
for prod in PRODUCTOS_AUTORIZADOS:
    m_prod = df_plata["producto_foco"] == prod
    # MCAR: Interpolación en variables climáticas
    df_plata.loc[m_prod, "precipitacion_acumulada_mm"] = (
        df_plata.loc[m_prod, "precipitacion_acumulada_mm"].interpolate().bfill().ffill()
    )
    # MAR: Mediana condicional por temporada en precios
    med_temp = df_plata.loc[m_prod].groupby("temporada")["precio_frecuente_kg"].transform("median")
    df_plata.loc[m_prod, "precio_frecuente_kg"] = df_plata.loc[m_prod, "precio_frecuente_kg"].fillna(med_temp)

print(f"Valores nulos tras imputación técnica: {df_plata.isna().sum().sum()}")


## 4. Tratamiento Contextual de Outliers y Ruido (Semana Santa vs Ruido)

> **Decisión Crítica de Ingeniería:** Un filtro de atípicos global ciego (como Z-Score > 3 o IQR global) eliminaría sistemáticamente los incrementos de precio de Cuaresma (+50%) y los picos por escasez de huracanes, destruyendo precisamente el fenómeno que el Observatorio debe predecir.

Se aplica un **IQR Estratificado por Especie y Temporada**:
$$	ext{IQR} = Q_3 - Q_1$$
- Los valores dentro de $1.5 	imes 	ext{IQR}$ se consideran variación ordinaria.
- En `TEMPORADA_ALTA_CUARESMA` y `TEMPORADA_HURACANES_LLUVIAS`, los valores altos se etiquetan como `es_pico_estacional_real` o `es_anomalia_climatica`.
- Únicamente los valores más allá de $3.0 	imes 	ext{IQR}$ (errores de captura) son sometidos a **Winsorización** acotada.


In [ ]:
df_plata["es_pico_estacional_real"] = False
df_plata["es_anomalia_climatica"] = False

for prod in PRODUCTOS_AUTORIZADOS:
    for temp in df_plata["temporada"].unique():
        mask = (df_plata["producto_foco"] == prod) & (df_plata["temporada"] == temp)
        if not mask.any():
            continue
        q1 = df_plata.loc[mask, "precio_frecuente_kg"].quantile(0.25)
        q3 = df_plata.loc[mask, "precio_frecuente_kg"].quantile(0.75)
        iqr = q3 - q1
        
        if temp == "TEMPORADA_ALTA_CUARESMA":
            df_plata.loc[mask & (df_plata["precio_frecuente_kg"] > q3), "es_pico_estacional_real"] = True
        elif temp == "TEMPORADA_HURACANES_LLUVIAS":
            df_plata.loc[mask & (df_plata["dias_cierre_puerto"] >= 2), "es_anomalia_climatica"] = True
            
        # Winsorización acotada a 3.0 * IQR (solo ruido severo)
        lim_sup = q3 + 3.0 * iqr
        lim_inf = max(10.0, q1 - 3.0 * iqr)
        df_plata.loc[mask, "precio_frecuente_kg"] = df_plata.loc[mask, "precio_frecuente_kg"].clip(lim_inf, lim_sup)

print("Outliers contextuales clasificados y tratados mediante IQR estratificado.")


## 5. Normalización, Estandarización y Jerarquías Conceptuales

Se transforman las variables continuas y se construyen variables categóricas de negocio:
1. **MinMax Scaling ($[0, 1]$):** Para algoritmos basados en funciones de distancia (KNN, Redes Neuronales).
2. **StandardScaler / Z-Score ($\mu=0, \sigma=1$):** Para modelos de regresión y series temporales (ARIMA, Regresión Lineal Regularizada).
3. **Jerarquías Conceptuales:**
   - `temporada_conceptual`: `ALTA_CUARESMA`, `BAJA_LLUVIAS`, `REGULAR`.
   - `nivel_escasez`: `BAJO`, `MEDIO`, `CRITICO` según días de cierre de puerto y alertas meteorológicas.
   - `nivel_precio_relativo`: `PRECIO_ACCESIBLE`, `PRECIO_MODERADO`, `PRECIO_ELEVADO` calibrado por especie.


In [ ]:
scaler_minmax = MinMaxScaler()
scaler_zscore = StandardScaler()

df_plata["precio_frecuente_minmax"] = scaler_minmax.fit_transform(df_plata[["precio_frecuente_kg"]]).round(4)
df_plata["precio_frecuente_zscore"] = scaler_zscore.fit_transform(df_plata[["precio_frecuente_kg"]]).round(4)
df_plata["precipitacion_minmax"] = scaler_minmax.fit_transform(df_plata[["precipitacion_acumulada_mm"]]).round(4)
df_plata["viento_zscore"] = scaler_zscore.fit_transform(df_plata[["velocidad_viento_kmh"]]).round(4)

# Jerarquía de temporada
df_plata["temporada_conceptual"] = df_plata["temporada"].map({
    "TEMPORADA_ALTA_CUARESMA": "ALTA_CUARESMA",
    "TEMPORADA_HURACANES_LLUVIAS": "BAJA_LLUVIAS",
    "TEMPORADA_REGULAR": "REGULAR"
})

# Nivel de Escasez
def clasificar_escasez(r):
    if r["dias_cierre_puerto"] >= 3 or r["alerta_meteorologica"] == "ROJA":
        return "CRITICO"
    elif r["dias_cierre_puerto"] >= 1 or r["alerta_meteorologica"] in ["AMARILLA", "NARANJA"]:
        return "MEDIO"
    return "BAJO"

df_plata["nivel_escasez"] = df_plata.apply(clasificar_escasez, axis=1)

print("Transformaciones y discretización completadas en Capa Plata.")
display(df_plata[["producto_foco", "precio_frecuente_kg", "precio_frecuente_minmax", "precio_frecuente_zscore", "nivel_escasez"]].head(3))


## 6. Reducción de Dimensionalidad y Construcción de Capa Oro

La **Capa Oro** consolida una vista analítica agregada a nivel mensual por especie, optimizada para responder directamente las preguntas analíticas del Observatorio:
- **E1 (Pronóstico):** Precio promedio, volatilidad ($\sigma$) y variación mensual.
- **E2 (Alertas vs Desabasto):** Conteo de alertas severas, días totales de puerto cerrado y semanas en escasez crítica.
- **E3 (Comparación Multivariada):** Coeficiente de variación (\%) e Índice Compuesto de Vulnerabilidad Climática.


In [ ]:
df_oro = df_plata.groupby(["mes", "producto_foco"], as_index=False).agg(
    periodos_observados=("periodo_id", "count"),
    temporada_dominante=("temporada_conceptual", lambda s: s.mode()[0]),
    precio_promedio_kg=("precio_frecuente_kg", lambda p: round(p.mean(), 2)),
    precio_volatilidad_std=("precio_frecuente_kg", lambda p: round(p.std(ddof=0), 2)),
    volumen_total_kg=("disponibilidad_volumen_kg", lambda v: round(v.sum(), 1)),
    precipitacion_acumulada_mm=("precipitacion_acumulada_mm", lambda pr: round(pr.sum(), 2)),
    dias_totales_cierre_puerto=("dias_cierre_puerto", "sum"),
    alertas_severas_count=("alerta_meteorologica", lambda a: (a.isin(["AMARILLA", "ROJA"])).sum()),
    semanas_escasez_critica=("nivel_escasez", lambda e: (e == "CRITICO").sum())
)

df_oro["coeficiente_variacion_precio_pct"] = round(
    (df_oro["precio_volatilidad_std"] / df_oro["precio_promedio_kg"]) * 100.0, 2
)

df_oro["indice_vulnerabilidad_climatica"] = round(
    (df_oro["dias_totales_cierre_puerto"] * 1.5) +
    (df_oro["alertas_severas_count"] * 1.0) +
    (df_oro["precipitacion_acumulada_mm"] / 50.0), 2
).clip(0.0, 10.0)

print(f"Capa Oro construida con éxito: {len(df_oro)} registros agregados.")
display(df_oro.head(5))


## 7. Conclusiones y Resumen de Métricas de Calidad E1

El pipeline de la entrega E1 demuestra la operatividad completa de un Trusted Data Lab:
1. **Segregación Confiable:** La Puerta de Calidad con Pandera aísla registros incompatibles en Cuarentena sin interrumpir la operación del pipeline.
2. **Preservación del Negocio:** El tratamiento de atípicos con IQR estratificado respeta las dinámicas de Cuaresma y huracanes.
3. **Listos para Modelado:** Los datasets en Capa Plata y Capa Oro proporcionan la base para entrenar los modelos predictivos de la entrega E2.
